# Building an RAG Pipeline Using PDF Chunking and Retrieval

## 1. Data Loader

In [11]:
from pathlib import Path
from typing import List, Any
from langchain_community.document_loaders import (
    PyPDFLoader, TextLoader, CSVLoader, Docx2txtLoader, JSONLoader
)

In [12]:
# Step 1 Loading the file
def load_all_documents(data_dir: str) -> List[Any]:
    """
        Load all supported fles from the data directory and convert to Langchain document structure
        Supported: PDF
        And then only we can convert them to chunking
    """
    # Use Project root data folder
    data_path = Path(data_dir).resolve()
    print(f"[DEBUG] Data Path: {data_path}")
    documents = []

    # PDF Files
    pdf_files = list(data_path.glob("**/*.pdf"))
    print(f"[DEBUG] Found {len(pdf_files)} PDF files: {[str(f) for f in pdf_files]}")
    for pdf_file in pdf_files:
        print(f"[DEBUG] Loading PDF: {pdf_file}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            loaded = loader.load()
            print(f"[DEBUG] Loaded {len(loaded)} PDF docs from {pdf_file}")
            documents.extend(loaded)
        except Exception as e:
            print(f"[ERROR] Failed to load PDF {pdf_file}: {e}")
            
    return documents

# all_pdf_documents = load_all_documents("../data")

## 2. Chunking & Embedding

In [13]:
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
import numpy as np

# Step 2
class EmbeddingPipeline:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2", chunk_size: int = 1000, chunk_overlap=200):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.model = SentenceTransformer(model_name)
        print(f"[INFO] Loaded embedding model: {model_name}")

    
    def chunk_documents(self, documents: List[Any]) -> List[str]:
        """ As soon as you get the document. You first chunk / Split it"""
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=self.chunk_size,
            chunk_overlap=self.chunk_overlap,
            length_function=len,
            separators=["\n\n", "\n", " ", ""]
        )
        chunks = splitter.split_documents(documents=documents)
        print(f"[INFO] Split {len(documents)} documents into {len(chunks)} chunks.")
        return chunks
    

    def embed_chunks(self, chunks: List[Any]) -> np.ndarray:
        """ Convert Chunks into Embedding Format (Vector Format) """
        texts = [chunk.page_content for chunk in chunks]
        print(f"[INFO] Generating embeddings for {len(texts)} chunks...")
        embeddings = self.model.encode(texts, show_progress_bar = True)
        print(f"[INFO] Embeddings shape: {embeddings.shape}")
        return embeddings


## FAISS VectorStore

``` python example FAISS code
import faiss
import numpy as np

dimension = 384
index = faiss.IndexFlatL2(dimension)

docs = ["Legal ethics", "Neural networks", "Space law in India"]
vectors = model.encode(docs).astype("float32")
index.add(vectors)

query_vector = model.encode(["Principles of AI ethics"]).astype("float32")
D, I = index.search(query_vector.reshape(1, -1), k=2)
print("Similar docs:", [docs[i] for i in I[0]])
```

In [14]:
import os
import uuid
import faiss
import numpy as np
import pickle
from typing import List, Any, Optional

class FaissVectorStore:
    def __init__(self, persist_dir: str = "faiss_store", embedding_model: str = "all-MiniLM-L6-v2", chunk_size = 1000, chunk_overlap = 200):
        self.persist_dir = persist_dir
        os.makedirs(self.persist_dir, exist_ok=True)
        self.index = None
        self.embedding_model = embedding_model
        self.model = SentenceTransformer(embedding_model)
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.metadata = []
        print(f"[INFO] Loaded embedding model: {embedding_model}")

    
    def build_from_documents(self, documents: List[Any]):
        print(f"[INFO] Building vector store from {len(documents)} raw documents...")
        emb_pipe = EmbeddingPipeline(model_name=self.embedding_model, chunk_size=self.chunk_size, chunk_overlap=self.chunk_overlap)
        
        chunks = emb_pipe.chunk_documents(documents=documents)
        embeddings = emb_pipe.embed_chunks(chunks=chunks)
        metadatas = [{"text": chunk.page_content} for chunk in chunks]
        
        self.add_embeddings(np.array(embeddings).astype("float32"), metadatas)
        self.save()
        print(f"[INFO] Vector store built and saved to {self.persist_dir}")


    def add_embeddings(self, embeddings: np.ndarray, metadatas: List[Any] = None):
        dim = embeddings.shape[1]
        if self.index is None:
            self.index = faiss.IndexFlatL2(dim)
        self.index.add(embeddings)
        if metadatas:
            self.metadata.extend(metadatas)
        print(f"[INFO] Vector store built and saved to {self.persist_dir}")


    def save(self):
        faiss_path = os.path.join(self.persist_dir, "faiss.index")
        meta_path = os.path.join(self.persist_dir, "metadata.pk1")
        
        faiss.write_index(self.index, faiss_path)
        with open(meta_path, "wb") as f:
            pickle.dump(self.metadata, f)
        print(f"[INFO] Loaded Faiss index and metadata from {self.persist_dir}")


    def load(self):
        faiss_path = os.path.join(self.persist_dir, "faiss.index")
        meta_path = os.path.join(self.persist_dir, "metadata.pk1")
        
        self.index = faiss.read_index(faiss_path)
        with open(meta_path, "rb") as f:
            self.metadata = pickle.load(f)
        print(f"[INFO] Loaded Faiss index and metadata from {self.persist_dir}")


    def search(self, query_embedding: np.ndarray, top_k: int = 5):
        D, I = self.index.search(query_embedding, top_k)
        results = []

        for idx, dist in zip(I[0], D[0]):
            meta = self.metadata[idx] if idx < len(self.metadata) else None
            results.append({"index": idx, "distance": dist, "metadata": meta})
        return results
    

    def query(self, query_text: str, top_k: int = 5):
        print(f"[INFO] Querying vector store for: '{query_text}'")
        query_emb = self.model.encode([query_text]).astype("float32")
        return self.search(query_emb, top_k=top_k)


## RAG Search

In [15]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq


load_dotenv()


class RAGSearch:
    def __init__(self, collection_name: str = "rag_documents", embedding_model: str = "all-MiniLM-L6-v2", llm_model: str = "llama-3.1-8b-instant"):
        # self.vectorstore = QdrantVectorStore(collection_name=collection_name, embedding_model=embedding_model)
        self.vectorstore = FaissVectorStore(persist_dir=collection_name, embedding_model=embedding_model)

        # Build the collection the first time (empty), otherwise just reuse what's in Qdrant
        # if self.vectorstore.count() == 0:
        docs = load_all_documents("../data")
        self.vectorstore.build_from_documents(docs)
        # else:
        self.vectorstore.load()

        groq_api_key = os.getenv("GROQ_API_KEY")
        self.llm = ChatGroq(
            api_key=groq_api_key, model=llm_model, temperature=0.1, max_tokens=1024
        )
        print(f"[INFO] Groq LLM intitalized: {llm_model}")

        

    def search_and_summarize(self, query: str, top_k: int = 5) -> str:
        results = self.vectorstore.query(query_text=query, top_k=top_k)
        texts = [r["metadata"].get("text", "") for r in results if r["metadata"]]
        context = "\n\n".join(texts)
        if not context:
            return "No relevant document found"

        prompt = f"""
            Summarize the following context for the query: '{query}
                Context: {context}
                Answer:
        """
        response = self.llm.invoke([prompt])
        return response.content


In [16]:
rag_search = RAGSearch()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1052.85it/s]


[INFO] Loaded embedding model: all-MiniLM-L6-v2
[DEBUG] Data Path: /app/LangChain/Pdf/data
[DEBUG] Found 3 PDF files: ['/app/LangChain/Pdf/data/pdf/statistics-canada.pdf', '/app/LangChain/Pdf/data/pdf/resume.pdf', '/app/LangChain/Pdf/data/pdf/annual_report_immigration.pdf']
[DEBUG] Loading PDF: /app/LangChain/Pdf/data/pdf/statistics-canada.pdf
[DEBUG] Loaded 26 PDF docs from /app/LangChain/Pdf/data/pdf/statistics-canada.pdf
[DEBUG] Loading PDF: /app/LangChain/Pdf/data/pdf/resume.pdf
[DEBUG] Loaded 3 PDF docs from /app/LangChain/Pdf/data/pdf/resume.pdf
[DEBUG] Loading PDF: /app/LangChain/Pdf/data/pdf/annual_report_immigration.pdf
[DEBUG] Loaded 57 PDF docs from /app/LangChain/Pdf/data/pdf/annual_report_immigration.pdf
[INFO] Building vector store from 86 raw documents...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1240.98it/s]


[INFO] Loaded embedding model: all-MiniLM-L6-v2
[INFO] Split 86 documents into 190 chunks.
[INFO] Generating embeddings for 190 chunks...


Batches: 100%|██████████| 6/6 [00:07<00:00,  1.23s/it]

[INFO] Embeddings shape: (190, 384)
[INFO] Vector store built and saved to rag_documents
[INFO] Loaded Faiss index and metadata from rag_documents
[INFO] Vector store built and saved to rag_documents
[INFO] Loaded Faiss index and metadata from rag_documents
[INFO] Groq LLM intitalized: llama-3.1-8b-instant


In [17]:
query = "Who is Vaishali?"
summary = rag_search.search_and_summarize(query=query, top_k=3)
print("Summary: ", summary)

[INFO] Querying vector store for: 'Who is Vaishali?'
Summary:  The context provided is about a professional named Vaishali Somu Ramesh, who is a Fullstack Developer with expertise in web development, data science, and machine learning. She has a strong educational background, having completed post-graduate studies in Computer Software and Database Development from Lambton College and a Bachelor of Engineering in BioInformatics from SRM University in India.


In [18]:
query = "Annual Report to Parliament on Immigration"
summary = rag_search.search_and_summarize(query=query, top_k=3)
print("Summary: ", summary)

[INFO] Querying vector store for: 'Annual Report to Parliament on Immigration'
Summary:  Here's a summary of the context for the query 'Annual Report to Parliament on Immigration' for the year 2025:

The Annual Report to Parliament on Immigration is a mandatory report that provides key admission highlights and data on immigration in Canada. The report is tabled in Parliament and is required by the Immigration and Refugee Protection Act (IRPA). For the year 2025, the report has been restructured to focus on key data tables, year-over-year comparisons, and metrics that are central to immigration planning and accountability. The report is produced by Immigration, Refugees and Citizenship Canada (IRCC) and is intended to inform Parliament and Canadians about immigration trends and data.


In [19]:
query = "Ombudsman Annual Report"
summary = rag_search.search_and_summarize(query=query, top_k=3)
print("Summary: ", summary)

[INFO] Querying vector store for: 'Ombudsman Annual Report'
Summary:  The context provided is the Annual Report 2024-2025 of the Office of the Ombudsman for the Department of National Defence and the Canadian Armed Forces. The report highlights various national engagements, website and social media statistics, and the Ombudsman's Advisory Committee. 

Key points from the report include:

1. National Engagements: The Ombudsman participated in several events, including Seamless Canada, Forum of Canadian Ombudsman, Ombuds Day Library of Parliament Seminar, and Canadian Institute for Military and Veteran Health Research Forum.

2. Website Statistics: There was a 5% increase in total page views, from 396,779 in 2023-2024 to 486,084 in 2024-2025.

3. Social Media Statistics: The Ombudsman has 3 social media channels, 6 social media accounts, and 7,549 followers, with an increase of 481 followers compared to the previous year.

4. Ombudsman's Advisory Committee: The Committee provided valuabl